# 🛰️ Project 1: Foundation Model Fine-Tuning for Crop Mapping

**Author:** Samuel Appiah Kubi  
**Programme:** Copernicus Master's in Digital Earth (Erasmus Mundus)  
**University:** Paris Lodron University Salzburg

---

## 📋 What This Notebook Does

This notebook walks through the **complete pipeline** for fine-tuning an Earth Observation foundation model to classify crop types (cocoa, oil palm, maize) from Sentinel-2 satellite imagery over Ghana.

| Step | What happens |
|------|-------------|
| 1️⃣ Setup | Install packages, clone repo, check GPU |
| 2️⃣ Data | Download Sentinel-2, cloud mask, compute indices |
| 3️⃣ Labels | Load/create crop type ground truth |
| 4️⃣ Dataset | Extract patches, spatial split, DataLoader |
| 5️⃣ Model | Load Clay / ResNet-50 foundation model |
| 6️⃣ Train | Two-phase fine-tuning with early stopping |
| 7️⃣ Evaluate | Metrics, confusion matrix, baseline comparison |
| 8️⃣ Map | Predict and export full crop type map |

---

### ⚙️ Runtime recommendation
> **Runtime → Change runtime type → T4 GPU** (free tier is sufficient)  
> Estimated training time on T4: **45–90 minutes** (depending on dataset size)


## 1️⃣ Environment Setup

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU available:')
    print(result.stdout[:500])
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
# This takes ~3 minutes on first run
!pip install -q \
    torch torchvision \
    rasterio geopandas shapely pyproj fiona rtree \
    eodag pystac \
    scikit-learn scikit-image \
    opencv-python-headless Pillow \
    pandas numpy scipy \
    folium plotly matplotlib seaborn \
    pyyaml tqdm transformers

# Optional: Clay EO foundation model
# !pip install -q git+https://github.com/Clay-foundation/model.git

print('\n✅ Installation complete')

In [ ]:
# ── Clone the pipeline repository ─────────────────────────────────────────
import os

REPO_PATH = '/content/crop_mapping'

if not os.path.exists(REPO_PATH):
    # Option A: Clone from GitHub (replace with your repo URL)
    # !git clone https://github.com/YOUR_USERNAME/project1_foundation_model.git {REPO_PATH}

    # Option B: Upload the folder via Files panel and copy here
    # For this notebook, we will create the directory structure inline
    os.makedirs(f'{REPO_PATH}/scripts', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/configs', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/raw', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/processed', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/labels', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/outputs/models', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/outputs/maps', exist_ok=True)
    os.makedirs(f'{REPO_PATH}/data/outputs/reports', exist_ok=True)
    print(f'Created directory structure at {REPO_PATH}')

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')
print('Contents:', os.listdir('.'))

In [ ]:
# ── Google Drive mounting (optional – for persistent storage) ─────────────
MOUNT_DRIVE = False   # Set to True to persist data across Colab sessions

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CropMappingProject'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f'Drive mounted. Project path: {DRIVE_PATH}')
else:
    print('Drive not mounted. Data will be lost when runtime resets.')
    print('Tip: Enable MOUNT_DRIVE=True to save models and data to Drive.')

## 2️⃣ Configuration

In [ ]:
# ── Write config.yaml ─────────────────────────────────────────────────────
config_content = """
study_area:
  name: "Ejura, Ashanti Region, Ghana"
  bbox: [7.15, -1.55, 7.60, -1.10]
  epsg: 32630

sentinel2:
  start_date: "2023-03-01"
  end_date:   "2023-10-31"
  cloud_cover_max: 20
  bands: ["B02", "B03", "B04", "B08", "B11"]
  resolution: 10
  product_type: "S2_MSI_L2A"
  provider: "cop_dataspace"
  max_scenes: 8
  scl_mask_classes: [0, 1, 3, 8, 9, 10]

indices:
  compute: ["NDVI", "NDWI", "EVI"]

patches:
  size: 32
  stride: 16
  bands: 8
  timesteps: 6

classes:
  names: ["maize", "cocoa", "oil_palm", "fallow", "other"]
  num_classes: 5
  label_map:
    maize: 0
    cocoa: 1
    oil_palm: 2
    fallow: 3
    other: 4
  colors:
    maize: "#FFD700"
    cocoa: "#8B4513"
    oil_palm: "#228B22"
    fallow: "#D2B48C"
    other: "#808080"

foundation_model:
  name: "resnet50"   # Change to 'clay' if Clay model available
  variant: "small"
  hf_model_id: "made-with-clay/Clay"
  fallback: "resnet50"
  input_channels: 8
  freeze_ratio: 0.85
  pretrained: true

training:
  epochs: 50
  batch_size: 16
  learning_rate_head: 1.0e-3
  learning_rate_backbone: 1.0e-5
  weight_decay: 1.0e-4
  optimizer: adamw
  scheduler: reduce_on_plateau
  patience: 10
  min_lr: 1.0e-6
  augmentation:
    random_flip: true
    random_rotation: [0, 90, 180, 270]
    brightness_jitter: 0.2
    contrast_jitter: 0.15
  class_weighting: true
  seed: 42

cross_validation:
  n_folds: 3
  strategy: spatial
  test_fold: 2

evaluation:
  metrics: [accuracy, precision, recall, f1, iou]
  confusion_matrix: true
  per_class: true

paths:
  raw:       data/raw
  processed: data/processed
  labels:    data/labels
  models:    data/outputs/models
  maps:      data/outputs/maps
  reports:   data/outputs/reports

baselines:
  random_forest:
    n_estimators: 200
    max_depth: 15
    n_jobs: -1
  unet_scratch:
    epochs: 20
    batch_size: 16
    lr: 1.0e-3
"""

with open('configs/config.yaml', 'w') as f:
    f.write(config_content)

import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('✅ Configuration loaded')
print(f"  Study area : {cfg['study_area']['name']}")
print(f"  Classes    : {cfg['classes']['names']}")
print(f"  Model      : {cfg['foundation_model']['name']}")
print(f"  Epochs     : {cfg['training']['epochs']}")

## 3️⃣ Sentinel-2 Data

> **Option A**: Download real Sentinel-2 data (requires Copernicus account)  
> **Option B**: Use synthetic data (instant, good for testing the pipeline)

In [ ]:
# ── Choose data mode ──────────────────────────────────────────────────────
USE_REAL_DATA = False   # Set True if you have Copernicus credentials

# If using real data, enter your Copernicus credentials:
COPERNICUS_USERNAME = ''   # Register at: https://dataspace.copernicus.eu/
COPERNICUS_PASSWORD = ''   # Free account

if USE_REAL_DATA:
    if not COPERNICUS_USERNAME:
        print('⚠️  Set COPERNICUS_USERNAME and COPERNICUS_PASSWORD above')
    else:
        import os
        os.environ['EODAG_USERNAME'] = COPERNICUS_USERNAME
        os.environ['EODAG_PASSWORD'] = COPERNICUS_PASSWORD
        from scripts.download_s2 import run as download_run
        download_run('configs/config.yaml', COPERNICUS_USERNAME, COPERNICUS_PASSWORD)
        print('✅ Real Sentinel-2 data downloaded')
else:
    print('Using synthetic data (demo mode)')
    print('→ Tip: For real results, set USE_REAL_DATA=True with Copernicus credentials')

In [ ]:
# ── Create synthetic Sentinel-2 composite (when USE_REAL_DATA=False) ──────
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import os

def create_synthetic_s2(cfg, seed=42):
    """Create a realistic-looking synthetic Sentinel-2 composite."""
    rng = np.random.default_rng(seed)
    bbox = cfg['study_area']['bbox']  # [min_lat, min_lon, max_lat, max_lon]
    H, W = 512, 512
    n_bands = cfg['patches']['bands']  # 5 spectral + 3 indices = 8

    # Simulate spectral patterns for different crop types
    data = np.zeros((n_bands, H, W), dtype=np.float32)

    # Create spatial blocks (simulating field patterns)
    block_h, block_w = H // 8, W // 8
    crop_signatures = {
        0: [0.03, 0.06, 0.02, 0.35, 0.18],  # maize: high NIR
        1: [0.02, 0.05, 0.02, 0.28, 0.14],  # cocoa: dense canopy
        2: [0.02, 0.05, 0.02, 0.32, 0.16],  # oil palm
        3: [0.05, 0.08, 0.04, 0.15, 0.10],  # fallow: bare soil
        4: [0.04, 0.07, 0.03, 0.20, 0.12],  # other
    }

    class_map = np.zeros((H, W), dtype=np.uint8)
    for bi in range(H // block_h):
        for bj in range(W // block_w):
            cls = rng.integers(0, 5)
            rs, re = bi * block_h, (bi + 1) * block_h
            cs, ce = bj * block_w, (bj + 1) * block_w
            class_map[rs:re, cs:ce] = cls
            sig = crop_signatures[cls]
            for band_idx, base_val in enumerate(sig):
                noise = rng.normal(0, base_val * 0.1, (re - rs, ce - cs))
                data[band_idx, rs:re, cs:ce] = base_val + noise

    # Add indices (bands 5-7)
    red, nir = data[2], data[3]
    green = data[1]
    blue = data[0]
    eps = 1e-8
    data[5] = np.clip((nir - red) / (nir + red + eps), -1, 1)   # NDVI
    data[6] = np.clip((green - nir) / (green + nir + eps), -1, 1)  # NDWI
    data[7] = np.clip(2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1 + eps), -1, 1)  # EVI

    transform = from_bounds(bbox[1], bbox[0], bbox[3], bbox[2], W, H)
    profile = {
        'driver': 'GTiff', 'dtype': 'float32', 'count': n_bands,
        'height': H, 'width': W, 'crs': CRS.from_epsg(4326),
        'transform': transform, 'nodata': -9999.0,
    }

    composite_path = 'data/processed/s2_composite.tif'
    os.makedirs('data/processed', exist_ok=True)
    with rasterio.open(composite_path, 'w', **profile) as dst:
        dst.write(data)

    return composite_path, class_map

composite_path, true_class_map = create_synthetic_s2(cfg)
print(f'✅ Synthetic composite created: {composite_path}')
print(f'   Shape: {true_class_map.shape}  Classes: {np.unique(true_class_map)}')

In [ ]:
# ── Visualise the composite ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import rasterio

with rasterio.open(composite_path) as src:
    composite = src.read()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RGB composite
rgb = np.stack([composite[2], composite[1], composite[0]], axis=-1)
lo, hi = np.percentile(rgb, [2, 98])
rgb_stretched = np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)
axes[0].imshow(rgb_stretched)
axes[0].set_title('RGB Composite (B4-B3-B2)', fontweight='bold')
axes[0].axis('off')

# NDVI
ndvi = composite[5]
im1 = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
axes[1].set_title('NDVI', fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# True class map
class_names = cfg['classes']['names']
colors = [cfg['classes']['colors'][n] for n in class_names]
cmap = ListedColormap(colors)
axes[2].imshow(true_class_map, cmap=cmap, vmin=0, vmax=4)
axes[2].set_title('True Crop Classes', fontweight='bold')
axes[2].axis('off')
patches = [mpatches.Patch(facecolor=c, label=n) for c, n in zip(colors, class_names)]
axes[2].legend(handles=patches, loc='lower left', fontsize=8)

plt.suptitle('Sentinel-2 Composite — Ejura, Ghana', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/outputs/reports/s2_composite_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Composite visualised')

## 4️⃣ Ground Truth Labels

In [ ]:
# ── Prepare labels ────────────────────────────────────────────────────────
from scripts.prepare_labels import run as prepare_labels_run

# For Colab demo: use synthetic labels
# For real use: change to 'cropharvest' or 'kml' with your file
LABEL_SOURCE = 'synthetic'

label_raster_path = prepare_labels_run(
    config_path='configs/config.yaml',
    source=LABEL_SOURCE,
)
print(f'\n✅ Labels prepared: {label_raster_path}')

In [ ]:
# ── Check label distribution ──────────────────────────────────────────────
import rasterio
import pandas as pd

label_raster_path = 'data/labels/label_raster.tif'
with rasterio.open(label_raster_path) as src:
    labels = src.read(1)

class_names = cfg['classes']['names']
counts = {}
for cls_id, name in enumerate(class_names):
    count = (labels == cls_id).sum()
    counts[name] = int(count)

df_counts = pd.DataFrame(list(counts.items()), columns=['Crop Class', 'Pixel Count'])
df_counts['%'] = (df_counts['Pixel Count'] / df_counts['Pixel Count'].sum() * 100).round(1)
print('Label distribution:')
print(df_counts.to_string(index=False))

# Visualise
colors = [cfg['classes']['colors'][n] for n in class_names]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Pixel Count')
ax.set_title('Crop Class Distribution in Labels', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for bar, count in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 5️⃣ Dataset and DataLoaders

In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────────────
from scripts.dataset import build_dataloaders, visualise_batch
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

train_loader, val_loader, test_loader, normaliser = build_dataloaders(cfg)

print(f'\n📦 DataLoaders ready')
print(f'  Train batches : {len(train_loader)}')
print(f'  Val batches   : {len(val_loader)}')
print(f'  Test batches  : {len(test_loader)}')

# Show a sample batch
x_sample, y_sample = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  x shape: {x_sample.shape}  (batch, channels, H, W)')
print(f'  y shape: {y_sample.shape}')
print(f'  x range: [{x_sample.min():.3f}, {x_sample.max():.3f}]')
print(f'  classes: {y_sample.unique().tolist()}')

In [ ]:
# ── Visualise sample patches ──────────────────────────────────────────────
from scripts.dataset import visualise_batch

x_vis = x_sample.numpy()
y_vis = y_sample.numpy()

visualise_batch(
    x_vis, y_vis,
    class_names=cfg['classes']['names'],
    n_samples=8,
    save_path='data/outputs/reports/sample_patches.png',
)

## 6️⃣ Foundation Model

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
from scripts.model import build_model, get_optimizer, get_scheduler, get_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(cfg).to(device)

# Test forward pass
x_test = torch.randn(4, cfg['patches']['bands'], cfg['patches']['size'], cfg['patches']['size']).to(device)
with torch.no_grad():
    out = model(x_test)
print(f'✅ Forward pass: input {x_test.shape} → output {out.shape}')

# Parameter summary
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable
print(f'\nModel parameter summary:')
print(f'  Total params     : {total:,}')
print(f'  Trainable (Phase 1): {trainable:,} ({100*trainable/total:.1f}%)')
print(f'  Frozen           : {frozen:,} ({100*frozen/total:.1f}%)')

## 7️⃣ Training

### Two-Phase Fine-Tuning

| Phase | Epochs | What's trained | Learning rate |
|-------|--------|----------------|---------------|
| **Phase 1** | 1 → 60% | Classification head only | 1e-3 (head) |
| **Phase 2** | 60% → end | Head + top backbone layers | 1e-5 (backbone), 1e-3 (head) |

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────
import json
import time
from IPython.display import clear_output

# Setup
train_ds = train_loader.dataset
class_weights = train_ds.class_weights.to(device)
criterion = get_loss(cfg, class_weights)
optimizer = get_optimizer(model, cfg)
scheduler = get_scheduler(optimizer, cfg)

# AMP for faster training on GPU
use_amp = device == 'cuda'
scaler = torch.cuda.amp.GradScaler() if use_amp else None

# Training state
total_epochs = cfg['training']['epochs']
phase2_start = int(total_epochs * 0.6)
patience = cfg['training']['patience']
best_val_loss = float('inf')
no_improve_count = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'phase': []}
phase2_started = False

print(f'Training for {total_epochs} epochs on {device}')
print(f'Phase 2 starts at epoch {phase2_start}')
print(f'Early stopping patience: {patience} epochs')
print('─' * 60)

start_time = time.time()

for epoch in range(total_epochs):
    # ── Phase 2 transition ─────────────────────────────────────────────────
    if epoch == phase2_start and not phase2_started:
        print(f'\n🔓 Phase 2 started at epoch {epoch+1}: unfreezing backbone layers')
        if hasattr(model, 'unfreeze_for_phase2'):
            model.unfreeze_for_phase2(n_transformer_blocks=2)
        elif hasattr(model, '_fallback') and hasattr(model._fallback, 'unfreeze_top_layers'):
            model._fallback.unfreeze_top_layers(2)
        optimizer = get_optimizer(model, cfg)
        scheduler = get_scheduler(optimizer, cfg)
        phase2_started = True

    phase = '2' if epoch >= phase2_start else '1'

    # ── Train one epoch ────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            from torch.cuda.amp import autocast
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            val_loss += criterion(logits, y).item()
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / max(total, 1)

    if scheduler:
        try: scheduler.step(val_loss)
        except: scheduler.step()

    # Record history
    history['train_loss'].append(float(train_loss))
    history['val_loss'].append(float(val_loss))
    history['val_acc'].append(float(val_acc))
    history['phase'].append(phase)

    # ── Save best ──────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_count = 0
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss, 'val_acc': val_acc, 'config': cfg,
        }, 'data/outputs/models/best_model.pth')
        star = '⭐'
    else:
        no_improve_count += 1
        star = ''

    lr = optimizer.param_groups[0]['lr']
    elapsed = (time.time() - start_time) / 60
    print(f'Ep {epoch+1:03d}/{total_epochs} [P{phase}] '
          f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.3f}  '
          f'lr={lr:.1e}  [{elapsed:.1f}min] {star}')

    # ── Early stopping ─────────────────────────────────────────────────────
    if no_improve_count >= patience:
        print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)')
        break

print(f'\n✅ Training complete! Best val_loss={best_val_loss:.4f}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────
from scripts.train import plot_training_history
plot_training_history(history, save_path='data/outputs/reports/training_curves.png')

## 8️⃣ Evaluation

In [ ]:
# ── Foundation model evaluation ───────────────────────────────────────────
from scripts.evaluate import evaluate_foundation_model, plot_confusion_matrix

print('Evaluating fine-tuned foundation model on TEST set…')
fm_metrics = evaluate_foundation_model(cfg, split='test')

if fm_metrics:
    plot_confusion_matrix(
        fm_metrics, cfg['classes']['names'], 'Foundation Model',
        save_path='data/outputs/reports/confusion_matrix_fm.png'
    )

In [ ]:
# ── Random Forest baseline ────────────────────────────────────────────────
from scripts.evaluate import evaluate_random_forest, compare_models

print('Training and evaluating Random Forest baseline…')
rf_metrics = evaluate_random_forest(cfg)

In [ ]:
# ── U-Net baseline ────────────────────────────────────────────────────────
from scripts.evaluate import train_unet_baseline

print('Training U-Net from scratch baseline…')
unet_metrics = train_unet_baseline(cfg)

In [ ]:
# ── Model comparison table ────────────────────────────────────────────────
all_metrics = [m for m in [fm_metrics, rf_metrics, unet_metrics] if m]
df_comparison = compare_models(
    all_metrics,
    save_path='data/outputs/reports/model_comparison.csv'
)

# Visual comparison
metrics_keys = ['F1 (macro)', 'IoU (macro)', 'Accuracy']
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(metrics_keys))
width = 0.25
colors_bar = ['royalblue', 'tomato', 'seagreen']

for i, row in df_comparison.iterrows():
    values = [float(row[k]) for k in metrics_keys]
    ax.bar(x + i * width, values, width, label=row['Model'],
           color=colors_bar[i % len(colors_bar)], alpha=0.85)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_keys)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.7, color='grey', linestyle='--', alpha=0.5, label='Target F1=0.7')
plt.tight_layout()
plt.savefig('data/outputs/reports/model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 9️⃣ Crop Type Map

In [ ]:
# ── Generate full classification map ──────────────────────────────────────
from scripts.predict_map import run as predict_run

outputs = predict_run('configs/config.yaml')
print('\nOutputs:')
for k, v in outputs.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Display interactive Folium map ────────────────────────────────────────
from scripts.predict_map import export_folium_map

if outputs.get('class_map'):
    html_path = export_folium_map(outputs['class_map'], cfg)
    from IPython.display import IFrame
    display(IFrame(html_path, width='100%', height=500))

## 🔟 Save and Export Results

In [ ]:
# ── Summary of all outputs ────────────────────────────────────────────────
import os

print('📁 Pipeline outputs:')
for root, dirs, files in os.walk('data/outputs'):
    level = root.replace('data/outputs', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f'{indent}  {file} ({size/1024:.0f} KB)')

In [ ]:
# ── Download results (Colab only) ─────────────────────────────────────────
import shutil
from google.colab import files

# Zip outputs for download
shutil.make_archive('/content/project1_outputs', 'zip', 'data/outputs')
print('Downloading outputs zip...')
files.download('/content/project1_outputs.zip')
print('✅ Download started')

## 📊 Results Interpretation

After completing the pipeline, interpret your results:

### F1 Score Guidelines
| F1 Score | Interpretation |
|----------|---------------|
| > 0.9 | Excellent — likely overfitting or easy classes |
| 0.7–0.9 | Good — model learned meaningful patterns |
| 0.5–0.7 | Moderate — acceptable for complex crop mapping |
| < 0.5 | Poor — check labels, increase data, tune LR |

### Common Issues
- **Low recall for rare class**: Add more labels for that class or increase class weight
- **Training loss drops but val loss doesn't**: Increase dropout or augmentation
- **Model predicts same class always**: Check class weights and label balance
- **OOM errors**: Reduce `batch_size` to 8 or `patch_size` to 16

### Next Steps
1. Run with real Sentinel-2 data and manually digitised field polygons
2. Experiment with Clay foundation model (`name: 'clay'` in config)
3. Try temporal stack as input (T, C, H, W) for better seasonal discrimination
4. Move to Project 2: Digital Twin for yield forecasting